<a href="https://colab.research.google.com/github/RyanDev1st/llm-and-engine/blob/feat%2Fchess-coach-sft/src/llm/llm_training/colab_e4b_qlora.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chess-Coach Agent — **Gemma 4 E4B** QLoRA on Google Colab (single T4)

Trains a LoRA adapter on the v1.2 agentic-harness SFT corpus (skills + tools + the
`python` verification tool; fast/think/auto reasoning modes), via **Unsloth** 4-bit
QLoRA so E4B fits one 16 GB T4.

**Run order:** Cell 1 (config) → 2 (GPU) → 3 (clone) → 4 (deps; *may need a runtime
restart*) → 5 (download base) → 6 (data check) → **6.5 (MEMORY PROBE — run before
training)** → 7 (train) → 8 (download adapter) → 9 (optional GGUF export).

**OOM is the enemy — this notebook is pessimistic about it:**
- 4-bit base + Unsloth gradient checkpointing + `paged_adamw_8bit` + batch=1.
- **Seq is fixed at 1664** (the data floor; lowering it truncates >50% of finals).
  If the probe (6.5) reports OOM, climb the ladder in Cell 1 — drop rank/targets,
  then fall to E2B — but **never cut seq**.

**Two T4s (Kaggle)?** DDP OOMs. Use ONE GPU, or `device_map="balanced"` to shard the
weights across both (model-parallel, not data-parallel). This notebook is single-T4.


In [ ]:
# Cell 1 - config (E4B QLoRA on ONE T4 — pessimistic / OOM-hardened)
import os

MODEL  = "gemma4_e4b"
ENGINE = "unsloth"   # Gemma-4-native: ~2x faster + ~70% less VRAM. The custom
                     # grounding-weighted loss + assistant mask still run (our loop,
                     # Unsloth kernels). "cuda" = stock HF fallback.

# Unsloth-published base (load_in_4bit). NOT a raw Google QAT checkpoint — that makes
# Unsloth re-init base k/v weights and train_unsloth.py fails LOUD (BaseReinitError).
HF_REPO = {
    ("gemma4_e4b", "unsloth"): "unsloth/gemma-4-E4B-it",   # confirmed current id (June 2026)
    ("gemma4_e2b", "unsloth"): "unsloth/gemma-4-E2B-it",   # fallback if E4B won't fit
    ("gemma4_e4b", "cuda"):    "google/gemma-4-E4B-it",
    ("gemma4_e2b", "cuda"):    "google/gemma-4-E2B-it",
}[(MODEL, ENGINE)]
NO_4BIT = False  # E4B MUST be 4-bit to fit 16 GB (Unsloth ~1-2% accuracy cost — recovered by QAT-style robustness)

CHESS_REPO_URL = "https://github.com/RyanDev1st/llm-and-engine.git"
CHESS_BRANCH = "feat/chess-coach-sft"
WORKDIR = "/content/llm-and-engine"
OUTPUT = "gemma4_chess_e4b_colab"

USE_DRIVE = True
DRIVE_DIR = "/content/drive/MyDrive/chess_coach_runs"

# SEQ is set by the DATA, not VRAM: corpus max row = 1655, MEDIAN = 1291. Below 1664
# the FINAL grounded answer is truncated off >50% of rows (a broken adapter). So seq
# is FIXED at 1664 — do NOT lower it to save memory.
MAX_SEQ = 1664
# OOM ladder (pessimistic — try the cheapest knob first, NEVER cut seq):
#   1) TARGETS "attn-only" + RANK 16  (start here — lightest useful adapter)
#   2) RANK 8                         (half the adapter VRAM)
#   3) GRAD_ACCUM up (32) only changes effective batch, not peak — safe to raise
#   4) LAST RESORT: MODEL="gemma4_e2b" (the fallback) — do NOT drop seq.
TARGETS = "attn-only"
RANK = 16
GRAD_ACCUM = 16          # effective batch = 1 x 16; raise freely (no VRAM cost)
MAX_STEPS = 120          # short DE-RISK; raise on the Kaggle 30h slot
EVAL_EVERY = 40
MAX_VAL = 128
print("config:", MODEL, "engine=", ENGINE, "base=", HF_REPO, "seq=", MAX_SEQ,
      "rank=", RANK, "targets=", TARGETS, "steps=", MAX_STEPS, "drive=", USE_DRIVE)


In [ ]:
# Cell 2 — GPU preflight + wall-time projection
import subprocess, torch
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,memory.free",
                      "--format=csv"], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), "No GPU — Runtime > Change runtime type > T4 GPU"
name = torch.cuda.get_device_name(0)
per = {"A100": 14, "L4": 32, "T4": 65}
sec = next((v for k, v in per.items() if k in name), 65)
print(f"device={name} | ~{sec}s/update -> {MAX_STEPS} steps ~= {MAX_STEPS*sec/3600:.1f}h "
      f"(raise MAX_STEPS if this is well under your session limit)")

In [ ]:
# Cell 3 — clone repo (code + data; skip LFS weights)
import os, subprocess

def run(cmd, **kw):
    print(">", " ".join(cmd)); return subprocess.run(cmd, check=True, text=True, **kw)

env = {**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"}
if not os.path.exists(WORKDIR):
    run(["git", "clone", "--depth", "1", "--branch", CHESS_BRANCH, CHESS_REPO_URL, WORKDIR], env=env)
else:
    run(["git", "-C", WORKDIR, "pull", "--ff-only"], env=env)
run(["git", "-C", WORKDIR, "log", "-1", "--oneline"])
os.environ["PYTHONPATH"] = f"{WORKDIR}/src/llm"
print("PYTHONPATH=", os.environ["PYTHONPATH"])

In [ ]:
# Cell 4 - deps. Let Unsloth resolve its own stack — do NOT pin an old transformers
# (Gemma 4 needs a current one; the old gemma-3n-era pins break E4B loading).
import subprocess, sys
def pip(*a): subprocess.run([sys.executable, "-m", "pip", "install", "-q", *a], check=True)

if ENGINE == "unsloth":
    # Unsloth pulls a compatible transformers/trl/peft/bnb/triton for the current
    # Colab torch. Keep it latest so Gemma 4 is recognized.
    pip("--upgrade", "unsloth", "unsloth_zoo", "bitsandbytes")
    pip("python-chess")
    # Do NOT import unsloth here — if the install bumped torch, Colab needs a runtime
    # restart first. The training subprocess imports it in a fresh process.
    print("unsloth installed (latest). If Colab shows a 'RESTART RUNTIME' banner, "
          "restart, then run Cells 5-9. If Gemma 4 fails to load with a model-type "
          "error, run:  pip install -U transformers  (Gemma 4 needs a current build).")
else:
    pip("-U", "transformers", "accelerate", "peft", "trl",
        "bitsandbytes", "datasets", "sentencepiece", "protobuf", "python-chess")
    import transformers, peft, bitsandbytes
    print("transformers", transformers.__version__, "| peft", peft.__version__,
          "| bnb", bitsandbytes.__version__)


In [ ]:
# Cell 5 — HF login (Colab Secrets) + download base model
import os
from huggingface_hub import login, snapshot_download
try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
except Exception:
    token = os.environ.get("HF_TOKEN")
assert token, "Add HF_TOKEN in the Colab Secrets panel (key icon) and enable notebook access."
login(token)
dest = f"{WORKDIR}/src/llm/models/{MODEL}"
snapshot_download(repo_id=HF_REPO, local_dir=dest,
                  allow_patterns=["*.json", "*.safetensors", "*.model", "*.txt", "tokenizer*"])
print("base model at", dest)
print(sorted(os.listdir(dest)))

In [ ]:
# Cell 6 — data check (REGENERATED qualitative corpus; stored gzipped)
import os, gzip
for name in ("v1_2_train.jsonl", "v1_2_val.jsonl"):
    base = f"{WORKDIR}/data/sft/{name}"
    path = base if os.path.exists(base) else base + ".gz"
    if not os.path.exists(path):
        n = 0
    elif path.endswith(".gz"):
        with gzip.open(path, "rt", encoding="utf-8") as fh:
            n = sum(1 for _ in fh)
    else:
        n = sum(1 for _ in open(path, encoding="utf-8"))
    print(name, "rows=", n, "(", os.path.basename(path), ")")
    assert n > 0, f"missing/empty {path} - pull the branch again"

In [ ]:
# Cell 6.5 — MEMORY PROBE: does E4B QLoRA fit ONE T4 at seq 1664? RUN BEFORE TRAINING.
# Loads 4-bit E4B + LoRA (fp16 on T4, auto), runs real backward steps at batch=1, prints
# PEAK VRAM + FIT/TIGHT/OOM. Uses the HEAVY config so a FIT verdict also covers lighter ones.
import subprocess, sys, os
base = f"{WORKDIR}/src/llm/models/{MODEL}"
print(f"probing E4B fit at seq={MAX_SEQ} targets={TARGETS} rank={RANK} ...")
subprocess.run([sys.executable, "-m", "llm_training.e4b_probe"],
               check=True, cwd=WORKDIR,
               env={**os.environ, "PYTHONPATH": f"{WORKDIR}/src/llm",
                    "CHESS_BASE": base, "PROBE_SEQ": str(MAX_SEQ),
                    "PROBE_TARGETS": TARGETS, "PROBE_RANK": str(RANK),
                    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True"})
# FIT/TIGHT -> run Cell 7. OOM -> climb the Cell-1 ladder (RANK 16->8, then E2B). NEVER lower seq.


In [ ]:
# Cell 7 — (optional) Drive for checkpoint persistence, then train. Drive is NON-FATAL:
# if mount fails (auth popup blocked/transient), fall back to local runs/ and keep going.
import subprocess, sys, os
mounted = False
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        os.makedirs(DRIVE_DIR, exist_ok=True)
        runs = f"{WORKDIR}/runs"
        if not os.path.islink(runs) and not os.path.exists(runs):
            os.symlink(DRIVE_DIR, runs)
        mounted = True
        print("checkpoints ->", DRIVE_DIR)
    except Exception as e:
        print(f"Drive mount failed ({e}); using LOCAL runs/ (download via Cell 8). "
              "To use Drive: re-run this cell and complete the auth popup, or set USE_DRIVE=False.")
if not mounted:
    os.makedirs(f"{WORKDIR}/runs", exist_ok=True)

cmd = [sys.executable, "-m", "llm_training.run_train",
       "--model", MODEL, "--engine", ENGINE, "--max-steps", str(MAX_STEPS), "--rank", str(RANK),
       "--targets", TARGETS, "--grad-accum", str(GRAD_ACCUM), "--max-seq", str(MAX_SEQ),
       "--eval-every", str(EVAL_EVERY), "--max-val", str(MAX_VAL),
       "--output", OUTPUT]
if NO_4BIT:
    cmd.append("--no-4bit")
print(">", " ".join(cmd))
subprocess.run(cmd, check=True, cwd=WORKDIR,
               env={**os.environ, "PYTHONPATH": f"{WORKDIR}/src/llm",
                    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True"})


In [ ]:
# Cell 8 — zip the adapter + download (also already on Drive if USE_DRIVE)
import shutil, os
run_dir = f"{WORKDIR}/runs/{OUTPUT}"
print("adapter files:", os.listdir(run_dir))
out_zip = f"/content/{OUTPUT}_adapter"
shutil.make_archive(out_zip, "zip", run_dir)
print("zip at", out_zip + ".zip")
try:
    from google.colab import files
    files.download(out_zip + ".zip")
except Exception as e:
    print("download from the Files panel instead:", out_zip + ".zip", "|", e)

## Cell 9 — serving: adapter → GGUF on the RTX 4060 (do this LOCALLY)

Colab's job is the **adapter** (Cell 8). GGUF conversion + serving happen on the 4060,
where llama.cpp lives. You always **merge (apply) the adapter into the base first**, then
convert — "GGUF vs apply-weight" is not either/or; merge is step 1 of producing the GGUF.

```bash
# on the 4060 box, after unzipping the adapter into runs/<OUTPUT>:
# 1) merge LoRA into the base (16-bit), then convert + quantize with llama.cpp
python -m llm_training.export_gguf runs/gemma4_chess_e4b_colab
#    -> writes a merged HF dir + a GGUF; serve via the bundled llama.cpp.
```

**Quant choice (matters — see below).** Gemma 4 GGUF via Unsloth currently supports only
**Q8_0 / BF16 / F16** (K-quants "later"). On the 4060 (8 GB):
- **Q8_0** — near-lossless, a 4B lands ~4–4.5 GB, fits with room. **Best default.**
- For a smaller file, convert to **F16** then `llama-quantize model.f16.gguf out.gguf Q5_K_M`
  (K-quant — far better than legacy `q4_0`, which this project already saw *fabricate eval
  numbers*; avoid `q4_0`).

Optional Colab export (may OOM on a T4 doing the 16-bit merge — prefer local):
`model.save_pretrained_gguf(out, tok, quantization_method="Q8_0")` after reloading the adapter.
